# Week 2 - Day 1 

This exercise will get 3 models talking to each other.

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# Load API keys
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

# Connect to Client Libraries
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# Models list
gpt_model = "gpt-4.1-mini"
ollama_model = "llama3.2"
openrouter_model = "nex-agi/nex-n2.5-mini:free"


OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-


In [2]:
# System prompts
gpt_system_prompt = """
You are Alvin, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Simon and Theodore.
"""

ollama_system_prompt = """
You are Simon, a know-it-all, arrogant chatbot. You try to agree sound important \
and prove that you know everything. You make sure that your point is final answer.
"""

or_system_prompt = """
You are Theodore, a very polite, courteous chatbot. You try to agree with \
everything the other people say, or find common ground. If the others are argumentative, \
you try to calm them down and keep chatting.
"""

In [3]:
# User prompts
def create_user_prompt(responder, bot2, bot3):
    return f"""
    You are {responder}, in conversation with {bot2} and {bot3}.
    Respond with what you would like to say next, as {responder}.
    The conversation so far is as follows:
    """

gpt_user_prompt_prefix = create_user_prompt("Alvin", "Simon", "Theodore")
ollama_user_prompt_prefix = create_user_prompt("Simon", "Alvin", "Theodore")
or_user_prompt_prefix = create_user_prompt("Theodore", "Simon", "Alvin")

In [4]:
def fill_in_converstion(userprompt, conversation):
    return f"""
    {userprompt}
    {conversation}"""

In [5]:
#gpt_user = fill_in_converstion(gpt_user_prompt, "This is the conversation")
#print(f"user: {gpt_user}")

In [6]:
def call_gpt(conversation):
    messages = [
        {"role": "system", "content": gpt_system_prompt},
        {"role": "user", "content": fill_in_converstion(gpt_user_prompt_prefix, conversation)}
        ]
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [7]:
def call_openrouter(conversation):
    messages = [
        {"role": "system", "content": or_system_prompt},
        {"role": "user", "content": fill_in_converstion(or_user_prompt_prefix, conversation)}
        ]
    response = openrouter.chat.completions.create(model=openrouter_model, messages=messages)
    return response.choices[0].message.content

In [8]:
def call_ollama(conversation):
    messages = [
        {"role": "system", "content": ollama_system_prompt},
        {"role": "user", "content": fill_in_converstion(ollama_user_prompt_prefix, conversation)}
        ]
    response = ollama.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
# Start the conversion
# gpt = Alvin
# ollama = Simon
# openrouter = Theodore

real_conversation = "Theodore: Hi."
print(real_conversation)

for i in range(2):
    gpt_says = " \n\nAlvin: " + call_gpt(real_conversation)
    print(gpt_says)
    real_conversation = real_conversation + gpt_says

    ollama_says = " \n\nSimon: " + call_ollama(real_conversation)
    print(ollama_says)
    real_conversation = real_conversation + ollama_says
    
    openrouter_says = " \n\nTheodore: " + call_openrouter(real_conversation)
    print(openrouter_says)
    real_conversation = real_conversation + openrouter_says